# Hardware-model-workload suitability framework

Purpose: convert the measured speed, latency, memory, quality, and execution status into a transparent practical recommendation framework for local deployment decisions.

This notebook reads only `results/processed/final-analysis-dataset.csv` and writes derived figures and tables under `analysis/`. Missing measurements are excluded from the relevant calculation rather than replaced with zero.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis").exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from analysis.utils import load_dataset, save_figure, save_table, successful, grouped_bar, add_display_hardware

data = load_dataset(PROJECT_ROOT / "results" / "processed" / "final-analysis-dataset.csv")
print(f"Loaded {len(data):,} rows and {len(data.columns):,} columns")

## Configurable decision thresholds

The thresholds below are explicit analytical assumptions, not universal definitions of suitability. They are placed at the top of the notebook so a dissertation sensitivity analysis can change them without rewriting the classification logic. A configuration must be successful and meet all suitable thresholds to be labelled **Suitable**. Successful configurations that meet the conditional floors but not every suitable threshold are **Conditionally suitable**. Failures and configurations below the conditional floors are **Not practical**.

In [ ]:
# Decision thresholds: change these values for sensitivity analysis and report the choice.
MIN_QUALITY_SUITABLE = 3.0       # 1-5 overall quality score
MIN_QUALITY_CONDITIONAL = 2.5    # 1-5 overall quality score
MIN_DECODE_SUITABLE = 2.0        # tokens/s
MIN_DECODE_CONDITIONAL = 0.5     # tokens/s
MAX_TTFT_SUITABLE = 1000.0       # recorded TTFT units
MAX_TTFT_CONDITIONAL = 5000.0    # recorded TTFT units
MAX_MEMORY_SUITABLE_MB = 12000.0  # relevant RAM/VRAM measure


## Configuration-level classification

Classification uses means over repeated executions for each hardware-model-workload combination. Missing quality or performance measures prevent a configuration from satisfying a threshold, rather than being silently treated as passing.

In [ ]:
framework = successful(data).copy()
framework["memory_usage_mb"] = framework["vram_usage"].where(framework["backend"].eq("cuda"), framework["ram_usage"])
configuration = framework.groupby(["hardware", "model", "workload"], as_index=False).agg(
    decode_tps=("decode_tps", "mean"),
    time_to_first_token=("time_to_first_token", "mean"),
    memory_usage_mb=("memory_usage_mb", "mean"),
    overall_score=("overall_score", "mean"),
    executions=("experiment_id", "size"),
    successes=("status", lambda s: s.eq("success").sum()),
)
configuration["quality_ok_suitable"] = configuration["overall_score"].ge(MIN_QUALITY_SUITABLE)
configuration["speed_ok_suitable"] = configuration["decode_tps"].ge(MIN_DECODE_SUITABLE)
configuration["latency_ok_suitable"] = configuration["time_to_first_token"].le(MAX_TTFT_SUITABLE)
configuration["memory_ok_suitable"] = configuration["memory_usage_mb"].le(MAX_MEMORY_SUITABLE_MB)
configuration["suitable"] = configuration[["quality_ok_suitable", "speed_ok_suitable", "latency_ok_suitable", "memory_ok_suitable"]].all(axis=1)
conditional = (
    configuration["overall_score"].ge(MIN_QUALITY_CONDITIONAL)
    & configuration["decode_tps"].ge(MIN_DECODE_CONDITIONAL)
    & configuration["time_to_first_token"].le(MAX_TTFT_CONDITIONAL)
)
configuration["suitability"] = "Not practical"
configuration.loc[conditional, "suitability"] = "Conditionally suitable"
configuration.loc[configuration["suitable"], "suitability"] = "Suitable"
display(configuration)
save_table(configuration, "06_configuration_suitability.csv")

## Hardware suitability matrix

The matrix counts suitable, conditional, and not-practical workload configurations for each hardware/model pair.

In [ ]:
hardware_matrix = pd.crosstab([configuration["hardware"], configuration["model"]], configuration["suitability"]).reset_index()
display(hardware_matrix)
save_table(hardware_matrix, "06_hardware_suitability_matrix.csv")

## Model suitability matrix

This matrix aggregates suitability across hardware and workload observations to identify models that are broadly deployable versus hardware-specific.

In [ ]:
model_matrix = pd.crosstab(configuration["model"], configuration["suitability"]).reset_index()
display(model_matrix)
save_table(model_matrix, "06_model_suitability_matrix.csv")

## Workload recommendation table

For each workload, the recommendation selects the highest-scoring suitable configuration when one exists, then the highest-scoring conditional configuration. This is a practical recommendation under the documented thresholds, not a claim that one configuration dominates every metric.

In [ ]:
rank_order = {"Suitable": 0, "Conditionally suitable": 1, "Not practical": 2}
recommendations = configuration.copy()
recommendations["suitability_rank"] = recommendations["suitability"].map(rank_order)
recommendations = recommendations.sort_values(
    ["workload", "suitability_rank", "overall_score", "decode_tps"],
    ascending=[True, True, False, False],
)
workload_recommendations = recommendations.groupby("workload", as_index=False).first()
workload_recommendations = workload_recommendations[["workload", "hardware", "model", "suitability", "overall_score", "decode_tps", "time_to_first_token", "memory_usage_mb"]]
display(workload_recommendations)
save_table(workload_recommendations, "06_workload_recommendations.csv")